# Knowledge Pilot

In [1]:
import json
from pathlib import Path

import torch
import yaml

from edu_eval.knowledge.indommlu import load_indommlu_csv
from edu_eval.knowledge.scorer import KnowledgeScorer
from edu_eval.models.loader import ModelLoader
from edu_eval.models.registry import ModelRegistry

In [2]:
PROJECT_ROOT = next(
    path
    for path in [Path.cwd(), *Path.cwd().parents]
    if (path / "pyproject.toml").exists()
)

MODEL_CONFIG_PATH = PROJECT_ROOT / "configs" / "models.yaml"
KNOWLEDGE_CONFIG_PATH = PROJECT_ROOT / "configs" / "knowledge.yaml"
DATA_PATH = PROJECT_ROOT / "data" / "raw" / "IndoMMLU.csv"
RESULTS_DIR = PROJECT_ROOT / "results" / "knowledge_pilot"

RESULTS_DIR.mkdir(parents=True, exist_ok=True)

with KNOWLEDGE_CONFIG_PATH.open("r", encoding="utf-8") as file:
    knowledge_config = yaml.safe_load(file)["knowledge"]

model_id = "smoke_qwen"

In [3]:
registry = ModelRegistry.from_yaml(MODEL_CONFIG_PATH)
model_spec = registry.get(model_id)

tokenizer, model, device = ModelLoader.load(model_spec)

print("Model :", model_spec.source)
print("Device:", device)
print("Dtype :", next(model.parameters()).dtype)

Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

Model : Qwen/Qwen2.5-0.5B-Instruct
Device: cuda
Dtype : torch.bfloat16


In [4]:
questions = load_indommlu_csv(DATA_PATH)

zero_shot_questions = [
    question
    for question in questions
    if not question.is_for_fewshot
]

primary_questions = [
    question
    for question in zero_shot_questions
    if question.level in knowledge_config["primary_levels"]
]

retention_questions = [
    question
    for question in zero_shot_questions
    if question.level in knowledge_config["retention_levels"]
]

print("Zero-shot :", len(zero_shot_questions))
print("Primary   :", len(primary_questions))
print("Retention :", len(retention_questions))

Zero-shot : 14904
Primary   : 8118
Retention : 4742


In [5]:
for level in ["SD", "SMP", "SMA"]:
    subset = [
        q
        for q in zero_shot_questions
        if q.level == level
    ]

    print(
        f"{level}: "
        f"{len(subset)} questions, "
        f"{len({q.subject for q in subset})} subjects"
    )

SD: 4520 questions, 18 subjects
SMP: 3598 questions, 18 subjects
SMA: 4742 questions, 21 subjects


In [6]:
scorer = KnowledgeScorer(
    model=model,
    tokenizer=tokenizer,
    device=device,
    span=knowledge_config.get("score_span", "full"),
)

import random

rng = random.Random(42)
pilot_questions = []

for level in ["SD", "SMP", "SMA"]:
    level_questions = [
        q
        for q in zero_shot_questions
        if q.level == level
    ]

    for subject in sorted({q.subject for q in level_questions}):
        subject_questions = [
            q
            for q in level_questions
            if q.subject == subject
        ]

        sample_size = 2 if level in {"SD", "SMP"} else 1

        pilot_questions.extend(
            rng.sample(
                subject_questions,
                min(sample_size, len(subject_questions)),
            )
        )

pilot_results = scorer.predict_many(
    pilot_questions,
    batch_size=knowledge_config["batch_size"],
)

print("Pilot size:", len(pilot_results))

Pilot size: 93


In [7]:
for level in ["SD", "SMP", "SMA"]:
    results = [
        r
        for r in pilot_results
        if r["level"] == level
    ]

    correct = sum(
        r["predictions"]["mean_log_likelihood"]["correct"]
        for r in results
    )

    accuracy = correct / len(results)

    print(
        f"{level}: "
        f"{accuracy:.3f} "
        f"({correct}/{len(results)})"
    )

SD: 0.333 (12/36)
SMP: 0.306 (11/36)
SMA: 0.286 (6/21)


In [8]:
print("\nSubject-group breakdown:")

groups = sorted(
    {
        q.subject_group
        for q in pilot_questions
    }
)

for group in groups:
    results = [
        r
        for r in pilot_results
        if next(
            q.subject_group
            for q in pilot_questions
            if q.question_id == r["question_id"]
        ) == group
    ]

    correct = sum(
        r["predictions"]["mean_log_likelihood"]["correct"]
        for r in results
    )

    print(
        f"{group:30s} "
        f"{correct}/{len(results)}"
    )


Subject-group breakdown:
Humanities                     10/26
Indonesian Language            1/5
Local Languages & Cultures     11/41
STEM                           4/9
Social science                 3/12


In [9]:
output_path = RESULTS_DIR / f"{model_id}_pilot.jsonl"

with output_path.open("w", encoding="utf-8") as file:
    for result in pilot_results:
        file.write(
            json.dumps(result, ensure_ascii=False) + "\n"
        )

print("Saved:", output_path)

Saved: /home/berdangry/Documents/edu-llm-evaluation/results/knowledge_pilot/smoke_qwen_pilot.jsonl


In [10]:
del scorer
del model
del tokenizer

if torch.cuda.is_available():
    torch.cuda.empty_cache()

print("Model resources released.")

Model resources released.
